# PREPROCESSING

In [ ]:
"""
preprocessing.py
================
Pipeline preprocessing dataset penyakit daun jagung.

Urutan tahapan:
    1. CLAHE               → perbaikan kontras adaptif lokal (channel L di Lab)
    2. Resize 224×224      → standarisasi ukuran input CNN
    3. Normalisasi [0, 1]  → konversi ke float32
    4. Split Dataset       → stratified split 80:10:10 (train:valid:test)
    5. Augmentasi          → target tetap 2000 gambar per kelas pada split train
    6. Export ZIP          → mengemas hasil akhir ke file .zip

Catatan:
    - Background removal TIDAK digunakan karena HSV thresholding rentan
      menghilangkan area penyakit (bercak kuning/coklat) yang penting.
      CNN modern mampu belajar membedakan area relevan secara otomatis.
    - Augmentasi HANYA diterapkan pada split train, bukan valid/test.
    - Output gambar disimpan sebagai file .jpg ke direktori output yang ditentukan.
    - File ZIP dibuat otomatis setelah seluruh pipeline selesai.
"""

import os
import random
import shutil
import zipfile
import cv2
import numpy as np
from datetime import datetime
from pathlib import Path
from tqdm import tqdm


# ─────────────────────────────────────────────────────────────────────────────
# KONFIGURASI PATH
# ─────────────────────────────────────────────────────────────────────────────

# Direktori dataset asli (sebelum preprocessing)
INPUT_DIR = '/kaggle/input/datasets/salmaaida/plantdisease-dataset-corn-ori-process/plantdiseasdataset_corn_ori_process_pict'

# Direktori output hasil preprocessing
OUTPUT_DIR = '/kaggle/working/preprocessed'

# Ukuran target resize (standar input CNN)
TARGET_SIZE = (224, 224)


# ─────────────────────────────────────────────────────────────────────────────
# KONFIGURASI FORMAT FILE GAMBAR
# ─────────────────────────────────────────────────────────────────────────────

# Set format file gambar yang didukung (mencakup variasi huruf besar/kecil)
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG'}


# ─────────────────────────────────────────────────────────────────────────────
# KONFIGURASI PARAMETER PREPROCESSING
# ─────────────────────────────────────────────────────────────────────────────

# --- CLAHE ---
# clip_limit : batas amplifikasi kontras (lebih tinggi = kontras lebih kuat)
# tile_grid_size : ukuran grid tile (lebih kecil = lebih lokal)
CLAHE_CLIP_LIMIT    = 2.0
CLAHE_TILE_GRID     = (8, 8)

# --- Output ---
# Kualitas JPEG saat menyimpan (1–100); 95 menjaga kualitas visual tinggi
JPEG_QUALITY = 95

# --- Split Dataset ---
# Direktori output hasil split train/valid/test
SPLIT_OUTPUT_DIR  = '/kaggle/working/split'

# Proporsi pembagian (train + valid + test harus = 1.0)
TRAIN_RATIO = 0.80
VALID_RATIO = 0.10
# TEST_RATIO  = 0.10  (implicit: 1 - TRAIN_RATIO - VALID_RATIO)

# Random seed untuk reprodusibilitas split dan augmentasi
RANDOM_SEED = 42

# --- Augmentasi ---
# Target jumlah gambar per kelas setelah augmentasi pada split train
AUG_TARGET_PER_CLASS = 2000

AUG_ROTATION_RANGE   = 20    # derajat maksimum rotasi (±)
AUG_BRIGHTNESS_RANGE = 0.15  # delta kecerahan proporsional (±15%)
AUG_ZOOM_RANGE       = 0.15  # zoom in maksimum (15%)
AUG_SHIFT_RANGE      = 0.10  # shift maksimum (±10% dimensi gambar)

# --- Export ZIP ---
# Direktori tempat file ZIP hasil akhir disimpan
ZIP_OUTPUT_DIR = '/kaggle/working'

# Nama file ZIP (None = dibuat otomatis dengan timestamp)
ZIP_FILENAME   = None

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FUNGSI HELPER
# ─────────────────────────────────────────────────────────────────────────────

def get_image_files(directory: Path) -> list:
    """
    Mengumpulkan semua file gambar dalam direktori berdasarkan ekstensi yang
    didukung (IMAGE_EXTENSIONS), mencakup variasi huruf besar dan kecil.

    Parameters
    ----------
    directory : Path
        Path direktori yang akan di-scan.

    Returns
    -------
    list
        Daftar Path objek untuk setiap file gambar yang ditemukan.
    """
    files = []
    for ext in IMAGE_EXTENSIONS:
        files.extend(directory.glob(f"*{ext}"))
    # Hapus duplikat yang mungkin muncul di sistem case-insensitive
    return list({f.resolve(): f for f in files}.values())

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FUNGSI PREPROCESSING
# ─────────────────────────────────────────────────────────────────────────────

def apply_clahe(image: np.ndarray) -> np.ndarray:
    """
    Menerapkan CLAHE (Contrast Limited Adaptive Histogram Equalization).

    Proses:
        - Konversi BGR → Lab color space
        - Terapkan CLAHE hanya pada channel L (Luminance/kecerahan)
        - Konversi kembali ke BGR

    Menggunakan Lab bukan HSV/YUV karena Lab memisahkan kecerahan
    dari informasi warna dengan lebih bersih, sehingga warna ciri
    penyakit (kuning, coklat, karat) tidak terdistorsi setelah enhancement.

    Parameters
    ----------
    image : np.ndarray
        Gambar input BGR (uint8).

    Returns
    -------
    np.ndarray
        Gambar dengan kontras yang diperbaiki, BGR uint8.
    """
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2Lab)
    l_channel, a_channel, b_channel = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=CLAHE_CLIP_LIMIT,
        tileGridSize=CLAHE_TILE_GRID
    )
    l_enhanced = clahe.apply(l_channel)

    lab_enhanced = cv2.merge([l_enhanced, a_channel, b_channel])
    result = cv2.cvtColor(lab_enhanced, cv2.COLOR_Lab2BGR)
    return result

In [ ]:
def resize_image(image: np.ndarray,
                 target_size: tuple = TARGET_SIZE) -> np.ndarray:
    """
    Resize gambar ke ukuran target menggunakan interpolasi LANCZOS4.

    LANCZOS4 digunakan karena memberikan kualitas downscaling yang lebih
    baik dibandingkan INTER_LINEAR atau INTER_AREA, meskipun lebih lambat.

    Parameters
    ----------
    image : np.ndarray
        Gambar input BGR (uint8).
    target_size : tuple
        Ukuran target (width, height) dalam piksel.

    Returns
    -------
    np.ndarray
        Gambar yang sudah di-resize, BGR uint8.
    """
    return cv2.resize(image, target_size, interpolation=cv2.INTER_LANCZOS4)

In [ ]:
def normalize_image(image: np.ndarray) -> np.ndarray:
    """
    Normalisasi nilai piksel ke range [0, 1].

    Mengonversi tipe data dari uint8 (0–255) ke float32 (0.0–1.0).
    Normalisasi dilakukan sebagai langkah terakhir karena fungsi OpenCV
    lainnya (CLAHE, filter, dll.) memerlukan input bertipe uint8.

    Parameters
    ----------
    image : np.ndarray
        Gambar input BGR (uint8, nilai 0–255).

    Returns
    -------
    np.ndarray
        Gambar ternormalisasi (float32, nilai 0.0–1.0).
    """
    return (image.astype(np.float32) / 255.0)

In [ ]:
def preprocess_image(image_path: str) -> np.ndarray | None:
    """
    Menjalankan pipeline preprocessing lengkap pada satu gambar.

    Urutan pipeline:
        1. CLAHE               (uint8)  → peningkatan kontras lokal
        2. Resize 224×224      (uint8)  → standarisasi ukuran input
        3. Normalisasi [0, 1]  (float32) ← SELALU TERAKHIR

    Catatan:
        Background removal TIDAK dilakukan karena HSV thresholding
        menghilangkan area penyakit (bercak kuning/coklat/karat) yang
        merupakan informasi kritis untuk klasifikasi penyakit daun jagung.

    Parameters
    ----------
    image_path : str
        Path lengkap ke file gambar.

    Returns
    -------
    np.ndarray or None
        Gambar hasil preprocessing (float32, shape: 224×224×3),
        atau None jika gambar gagal dibaca.
    """
    # Baca gambar
    image = cv2.imread(image_path)
    if image is None:
        print(f"  [SKIP] Gagal membaca: {image_path}")
        return None

    # ── Step 1: CLAHE ─────────────────────────────────────────────────────────
    image = apply_clahe(image)

    # ── Step 2: Resize ke 224×224 ─────────────────────────────────────────────
    image = resize_image(image)

    # ── Step 3: Normalisasi [0, 1] (float32) ──────────────────────────────────
    image = normalize_image(image)

    return image

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FUNGSI SIMPAN GAMBAR
# ─────────────────────────────────────────────────────────────────────────────

def save_preprocessed_image(image_normalized: np.ndarray,
                             output_path: str) -> bool:
    """
    Menyimpan gambar hasil preprocessing (float32 [0,1]) ke file JPEG.

    Konversi kembali ke uint8 (0–255) sebelum disimpan karena cv2.imwrite
    tidak mendukung format float32 secara langsung.

    Parameters
    ----------
    image_normalized : np.ndarray
        Gambar float32 dengan nilai [0, 1].
    output_path : str
        Path file output (harus berekstensi .jpg atau .png).

    Returns
    -------
    bool
        True jika berhasil disimpan, False jika gagal.
    """
    # Konversi float32 [0,1] → uint8 [0,255] untuk disimpan
    image_uint8 = (image_normalized * 255.0).clip(0, 255).astype(np.uint8)

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    success = cv2.imwrite(
        output_path,
        image_uint8,
        [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY]
    )
    return success

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FUNGSI UTAMA: PROSES SELURUH DATASET
# ─────────────────────────────────────────────────────────────────────────────

def preprocess_dataset(input_dir: str, output_dir: str) -> dict:
    """
    Memproses seluruh dataset secara batch.

    Struktur direktori input yang diharapkan:
        input_dir/
        ├── Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot/
        │   ├── image1.jpg
        │   └── ...
        ├── Corn_(maize)___Common_rust_/
        ├── Corn_(maize)___Northern_Leaf_Blight/
        └── Corn_(maize)___healthy/

    Struktur direktori output yang dihasilkan (sama dengan input):
        output_dir/
        ├── Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot/
        ├── Corn_(maize)___Common_rust_/
        ├── ...

    Format file gambar yang didukung: .jpg, .jpeg, .png (dan variasinya
    dalam huruf besar: .JPG, .JPEG, .PNG).

    Parameters
    ----------
    input_dir : str
        Path direktori dataset asli.
    output_dir : str
        Path direktori untuk menyimpan hasil preprocessing.

    Returns
    -------
    dict
        Ringkasan hasil: jumlah berhasil, gagal, dan total per kelas.
    """
    input_path  = Path(input_dir)
    output_path = Path(output_dir)

    if not input_path.exists():
        raise FileNotFoundError(f"Direktori input tidak ditemukan: {input_dir}")

    # Kumpulkan semua subdirektori kelas
    class_dirs = [d for d in input_path.iterdir() if d.is_dir()]

    if not class_dirs:
        raise ValueError(f"Tidak ada subdirektori kelas di: {input_dir}")

    print(f"\n{'='*60}")
    print(f"  PIPELINE PREPROCESSING DATASET PENYAKIT DAUN JAGUNG")
    print(f"{'='*60}")
    print(f"  Input  : {input_dir}")
    print(f"  Output : {output_dir}")
    print(f"  Target : {TARGET_SIZE[0]}×{TARGET_SIZE[1]} piksel")
    print(f"  Format : {', '.join(sorted(IMAGE_EXTENSIONS))}")
    print(f"  Kelas  : {len(class_dirs)} kelas ditemukan")
    print(f"{'='*60}\n")

    summary = {
        "total_success": 0,
        "total_failed":  0,
        "per_class":     {}
    }

    for class_dir in sorted(class_dirs):
        class_name  = class_dir.name
        # Gunakan helper get_image_files untuk mencakup semua ekstensi
        image_files = get_image_files(class_dir)

        if not image_files:
            print(f"[INFO] Tidak ada gambar di kelas: {class_name}\n")
            continue

        print(f"[KELAS] {class_name}")
        print(f"        {len(image_files)} gambar ditemukan")

        success_count = 0
        failed_count  = 0

        out_class_dir = output_path / class_name
        out_class_dir.mkdir(parents=True, exist_ok=True)

        for img_file in tqdm(image_files, desc=f"  Preprocessing", unit="img"):
            # Buat path output (pertahankan nama file asli, ubah ekstensi ke .jpg)
            out_filename = img_file.stem + ".jpg"
            out_filepath = str(out_class_dir / out_filename)

            # Skip jika sudah diproses sebelumnya
            if os.path.exists(out_filepath):
                success_count += 1
                continue

            # Jalankan pipeline preprocessing
            result = preprocess_image(str(img_file))

            if result is None:
                failed_count += 1
                continue

            # Simpan hasil
            saved = save_preprocessed_image(result, out_filepath)
            if saved:
                success_count += 1
            else:
                print(f"  [ERROR] Gagal menyimpan: {out_filepath}")
                failed_count += 1

        summary["total_success"] += success_count
        summary["total_failed"]  += failed_count
        summary["per_class"][class_name] = {
            "total":   len(image_files),
            "success": success_count,
            "failed":  failed_count
        }

        print(f"        Berhasil: {success_count} | Gagal: {failed_count}\n")

    # Tampilkan ringkasan akhir
    _print_summary(summary)
    return summary

In [ ]:
def _print_summary(summary: dict) -> None:
    """Mencetak ringkasan hasil preprocessing ke konsol."""
    print(f"\n{'='*60}")
    print(f"  RINGKASAN HASIL PREPROCESSING")
    print(f"{'='*60}")
    print(f"  {'Kelas':<50} {'Total':>6} {'OK':>6} {'Gagal':>6}")
    print(f"  {'-'*50} {'------':>6} {'------':>6} {'------':>6}")

    for class_name, stats in summary["per_class"].items():
        # Pendekan nama kelas agar muat di tabel
        short_name = class_name[:48] + ".." if len(class_name) > 48 else class_name
        print(
            f"  {short_name:<50} "
            f"{stats['total']:>6} "
            f"{stats['success']:>6} "
            f"{stats['failed']:>6}"
        )

    print(f"  {'-'*50} {'------':>6} {'------':>6} {'------':>6}")
    total = sum(s["total"]   for s in summary["per_class"].values())
    ok    = summary["total_success"]
    fail  = summary["total_failed"]
    print(f"  {'TOTAL':<50} {total:>6} {ok:>6} {fail:>6}")
    print(f"{'='*60}\n")

    if fail == 0:
        print("  ✅ Semua gambar berhasil diproses!")
    else:
        print(f"  ⚠️  {fail} gambar gagal diproses. Cek log di atas.")
    print()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FUNGSI SPLIT DATASET
# ─────────────────────────────────────────────────────────────────────────────

def split_dataset(preprocessed_dir: str, split_dir: str,
                  train_ratio: float = TRAIN_RATIO,
                  valid_ratio: float = VALID_RATIO,
                  seed: int = RANDOM_SEED) -> dict:
    """
    Membagi dataset hasil preprocessing ke split train/valid/test
    secara stratified (proporsi kelas terjaga di setiap split).

    Skema pembagian default:
        - Train : 80%
        - Valid  : 10%
        - Test   : 10%

    Stratified split memastikan distribusi kelas pada setiap split
    mencerminkan distribusi kelas pada dataset secara keseluruhan,
    sehingga tidak ada split yang didominasi satu kelas tertentu.

    Struktur output:
        split_dir/
        ├── train/
        │   ├── Corn_(maize)___Common_rust_/
        │   └── ...
        ├── valid/
        │   └── ...
        └── test/
            └── ...

    Parameters
    ----------
    preprocessed_dir : str
        Path direktori hasil preprocessing (output dari preprocess_dataset).
    split_dir : str
        Path direktori output untuk menyimpan hasil split.
    train_ratio : float
        Proporsi data training (default: 0.80).
    valid_ratio : float
        Proporsi data validasi (default: 0.10).
    seed : int
        Random seed untuk reprodusibilitas (default: 42).

    Returns
    -------
    dict
        Ringkasan jumlah gambar per split per kelas.
    """
    from sklearn.model_selection import train_test_split as _split

    preprocessed_path = Path(preprocessed_dir)
    split_path        = Path(split_dir)

    if not preprocessed_path.exists():
        raise FileNotFoundError(
            f"Direktori hasil preprocessing tidak ditemukan: {preprocessed_dir}"
        )

    # Kumpulkan semua file beserta labelnya
    all_files  = []
    all_labels = []
    class_dirs = sorted([d for d in preprocessed_path.iterdir() if d.is_dir()])

    for class_dir in class_dirs:
        # Gunakan helper get_image_files untuk mencakup semua ekstensi
        img_files = get_image_files(class_dir)
        for img_file in img_files:
            all_files.append(img_file)
            all_labels.append(class_dir.name)

    if not all_files:
        raise ValueError(
            f"Tidak ada file gambar ditemukan di: {preprocessed_dir}"
        )

    test_ratio = round(1.0 - train_ratio - valid_ratio, 10)
    n_total    = len(all_files)

    print(f"\n{'='*60}")
    print(f"  SPLIT DATASET (STRATIFIED)")
    print(f"{'='*60}")
    print(f"  Total gambar  : {n_total}")
    print(f"  Train         : {train_ratio*100:.0f}%  "
          f"(≈ {int(n_total * train_ratio)} gambar)")
    print(f"  Valid         : {valid_ratio*100:.0f}%  "
          f"(≈ {int(n_total * valid_ratio)} gambar)")
    print(f"  Test          : {test_ratio*100:.0f}%  "
          f"(≈ {int(n_total * test_ratio)} gambar)")
    print(f"  Random seed   : {seed}")
    print(f"  Output        : {split_dir}")
    print(f"{'='*60}\n")

    # ── Split 1: train vs (valid + test) ──────────────────────────────────────
    X_train, X_temp, y_train, y_temp = _split(
        all_files, all_labels,
        test_size=round(1.0 - train_ratio, 10),
        stratify=all_labels,
        random_state=seed
    )

    # ── Split 2: valid vs test (dari sisa X_temp) ─────────────────────────────
    # valid_ratio / (valid_ratio + test_ratio) = proporsi valid dari temp
    valid_from_temp = valid_ratio / (1.0 - train_ratio)
    X_valid, X_test, y_valid, y_test = _split(
        X_temp, y_temp,
        test_size=round(1.0 - valid_from_temp, 10),
        stratify=y_temp,
        random_state=seed
    )

    # ── Salin file ke direktori split ─────────────────────────────────────────
    splits = {
        "train": (X_train, y_train),
        "valid": (X_valid, y_valid),
        "test":  (X_test,  y_test),
    }

    for split_name, (files, labels) in splits.items():
        print(f"  Menyalin split '{split_name}' ({len(files)} gambar)...")
        for file_path, label in tqdm(
            zip(files, labels), total=len(files),
            desc=f"  {split_name:>5}", unit="file"
        ):
            dest_dir = split_path / split_name / label
            dest_dir.mkdir(parents=True, exist_ok=True)
            shutil.copy2(file_path, dest_dir / file_path.name)

    # ── Hitung ringkasan ───────────────────────────────────────────────────────
    summary = {"train": {}, "valid": {}, "test": {}}
    for split_name in ["train", "valid", "test"]:
        for class_dir in class_dirs:
            dest  = split_path / split_name / class_dir.name
            # Hitung semua ekstensi yang didukung
            count = len(get_image_files(dest)) if dest.exists() else 0
            summary[split_name][class_dir.name] = count

    _print_split_summary(summary, n_total)
    return summary

In [ ]:
def _print_split_summary(summary: dict, total_images: int) -> None:
    """Mencetak ringkasan hasil split ke konsol."""
    splits      = ["train", "valid", "test"]
    class_names = list(next(iter(summary.values())).keys())

    print(f"\n{'='*72}")
    print(f"  RINGKASAN SPLIT DATASET")
    print(f"{'='*72}")
    print(f"  {'Kelas':<44} {'Train':>7} {'Valid':>7} {'Test':>7} {'Total':>7}")
    print(f"  {'-'*44} {'-------':>7} {'-------':>7} {'-------':>7} {'-------':>7}")

    for class_name in class_names:
        short  = class_name[:42] + ".." if len(class_name) > 42 else class_name
        counts = [summary[s].get(class_name, 0) for s in splits]
        print(
            f"  {short:<44} "
            f"{counts[0]:>7} {counts[1]:>7} {counts[2]:>7} {sum(counts):>7}"
        )

    print(f"  {'-'*44} {'-------':>7} {'-------':>7} {'-------':>7} {'-------':>7}")
    totals = [sum(summary[s].values()) for s in splits]
    print(
        f"  {'TOTAL':<44} "
        f"{totals[0]:>7} {totals[1]:>7} {totals[2]:>7} {sum(totals):>7}"
    )
    print(f"{'='*72}\n")
    print(f"  ✅ Split selesai! ({sum(totals)} gambar dari {total_images} total)")
    print()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FUNGSI AUGMENTASI
# ─────────────────────────────────────────────────────────────────────────────

# ── Fungsi augmentasi individual ──────────────────────────────────────────────

def aug_flip_horizontal(image: np.ndarray) -> np.ndarray:
    """
    Membalik gambar secara horizontal (kiri ↔ kanan).

    Relevan untuk dataset lahan jagung karena orientasi daun
    di lapangan dapat bervariasi ke kiri maupun ke kanan.
    """
    return cv2.flip(image, 1)


def aug_flip_vertical(image: np.ndarray) -> np.ndarray:
    """
    Membalik gambar secara vertikal (atas ↔ bawah).

    Menangkap perspektif pengambilan gambar dari berbagai sudut
    di lahan jagung (kamera atas maupun samping).
    """
    return cv2.flip(image, 0)


def aug_rotate(image: np.ndarray,
               max_angle: float = AUG_ROTATION_RANGE) -> np.ndarray:
    """
    Memutar gambar dengan sudut acak dalam rentang [-max_angle, +max_angle].

    Mensimulasikan variasi sudut pengambilan gambar di lahan jagung.
    Piksel kosong akibat rotasi diisi metode BORDER_REFLECT agar tidak ada
    area hitam di tepi gambar.

    Parameters
    ----------
    image : np.ndarray
        Gambar input BGR (uint8).
    max_angle : float
        Sudut maksimum rotasi dalam derajat (default: AUG_ROTATION_RANGE).
    """
    angle = random.uniform(-max_angle, max_angle)
    h, w  = image.shape[:2]
    M     = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    return cv2.warpAffine(image, M, (w, h), borderMode=cv2.BORDER_REFLECT)


def aug_brightness(image: np.ndarray,
                   max_delta: float = AUG_BRIGHTNESS_RANGE) -> np.ndarray:
    """
    Menyesuaikan kecerahan gambar dengan delta acak ±max_delta * 255.

    Mensimulasikan kondisi pencahayaan yang berbeda di lapangan
    (pagi, siang, mendung) pada lahan jagung.
    Nilai piksel di-clip ke [0, 255] setelah penyesuaian.

    Parameters
    ----------
    image : np.ndarray
        Gambar input BGR (uint8).
    max_delta : float
        Delta kecerahan proporsional (default: AUG_BRIGHTNESS_RANGE = ±15%).
    """
    delta  = random.uniform(-max_delta, max_delta)
    result = image.astype(np.float32) + (delta * 255.0)
    return np.clip(result, 0, 255).astype(np.uint8)


def aug_zoom(image: np.ndarray,
             max_zoom: float = AUG_ZOOM_RANGE) -> np.ndarray:
    """
    Melakukan zoom in secara acak (antara 1.0 hingga 1 + max_zoom kali).

    Mensimulasikan variasi jarak kamera ke daun jagung saat pengambilan
    gambar di lapangan.

    Proses:
        - Potong area gambar secara acak (random crop)
        - Resize kembali ke ukuran semula dengan LANCZOS4

    Parameters
    ----------
    image : np.ndarray
        Gambar input BGR (uint8).
    max_zoom : float
        Zoom in maksimum (default: AUG_ZOOM_RANGE = 15%).
    """
    zoom  = random.uniform(1.0, 1.0 + max_zoom)
    h, w  = image.shape[:2]
    new_h = int(h / zoom)
    new_w = int(w / zoom)
    y0    = random.randint(0, max(0, h - new_h))
    x0    = random.randint(0, max(0, w - new_w))
    crop  = image[y0:y0 + new_h, x0:x0 + new_w]
    return cv2.resize(crop, (w, h), interpolation=cv2.INTER_LANCZOS4)


def aug_shift(image: np.ndarray,
              max_shift: float = AUG_SHIFT_RANGE) -> np.ndarray:
    """
    Menggeser (translate) gambar secara acak dalam rentang ±max_shift
    dari dimensi gambar (horizontal dan vertikal).

    Mensimulasikan variasi posisi objek daun di dalam frame kamera.
    Piksel kosong diisi metode BORDER_REFLECT.

    Parameters
    ----------
    image : np.ndarray
        Gambar input BGR (uint8).
    max_shift : float
        Shift maksimum proporsional (default: AUG_SHIFT_RANGE = ±10%).
    """
    h, w = image.shape[:2]
    tx   = random.uniform(-max_shift, max_shift) * w
    ty   = random.uniform(-max_shift, max_shift) * h
    M    = np.float32([[1, 0, tx], [0, 1, ty]])
    return cv2.warpAffine(image, M, (w, h), borderMode=cv2.BORDER_REFLECT)


def aug_gaussian_noise(image: np.ndarray,
                       std: float = 0.02) -> np.ndarray:
    """
    Menambahkan Gaussian noise pada gambar.

    Mensimulasikan noise sensor kamera di kondisi lapangan yang
    beragam (misal: gambar pada resolusi rendah atau kamera bergerak).

    Parameters
    ----------
    image : np.ndarray
        Gambar input BGR (uint8).
    std : float
        Standar deviasi noise (proporsional terhadap [0,255]).
        Default: 0.02 → std = 5.1 pada skala 0–255.
    """
    noise  = np.random.normal(0, std * 255.0, image.shape).astype(np.float32)
    result = image.astype(np.float32) + noise
    return np.clip(result, 0, 255).astype(np.uint8)


def aug_gamma_correction(image: np.ndarray,
                          gamma_range: tuple = (0.7, 1.4)) -> np.ndarray:
    """
    Menerapkan koreksi gamma acak pada gambar.

    Mensimulasikan variasi kondisi pencahayaan lingkungan pada lahan jagung.
    Gamma < 1.0 mencerahkan gambar, gamma > 1.0 menggelapkan gambar.

    Parameters
    ----------
    image : np.ndarray
        Gambar input BGR (uint8).
    gamma_range : tuple
        Rentang nilai gamma (min, max). Default: (0.7, 1.4).
    """
    gamma     = random.uniform(*gamma_range)
    inv_gamma = 1.0 / gamma
    table     = np.array([
        ((i / 255.0) ** inv_gamma) * 255
        for i in range(256)
    ], dtype=np.uint8)
    return cv2.LUT(image, table)


# Pool semua fungsi augmentasi yang tersedia
_AUGMENTATION_POOL = [
    aug_flip_horizontal,
    aug_flip_vertical,
    aug_rotate,
    aug_brightness,
    aug_zoom,
    aug_shift,
    aug_gaussian_noise,
    aug_gamma_correction,
]

In [ ]:
def apply_random_augmentation(image: np.ndarray,
                               min_ops: int = 1,
                               max_ops: int = 3) -> np.ndarray:
    """
    Menerapkan kombinasi augmentasi acak pada sebuah gambar.

    Secara acak memilih antara min_ops hingga max_ops fungsi dari
    _AUGMENTATION_POOL dan menerapkannya secara berurutan.

    Parameters
    ----------
    image : np.ndarray
        Gambar input BGR (uint8).
    min_ops : int
        Jumlah minimum fungsi augmentasi yang diterapkan (default: 1).
    max_ops : int
        Jumlah maksimum fungsi augmentasi yang diterapkan (default: 3).

    Returns
    -------
    np.ndarray
        Gambar hasil augmentasi (BGR, uint8).
    """
    n_ops    = random.randint(min_ops, min(max_ops, len(_AUGMENTATION_POOL)))
    selected = random.sample(_AUGMENTATION_POOL, n_ops)
    result   = image.copy()
    for aug_fn in selected:
        result = aug_fn(result)
    return result

In [ ]:
def augment_train_data(split_dir: str,
                         target_per_class: int = AUG_TARGET_PER_CLASS,
                         seed: int = RANDOM_SEED) -> dict:
    """
    Melakukan augmentasi pada data train hingga mencapai target jumlah
    gambar yang ditentukan (fixed target per kelas).

    Strategi (fixed target):
        - Hitung jumlah gambar per kelas di split train
        - Target count = AUG_TARGET_PER_CLASS (default: 2000) untuk SEMUA kelas
        - Untuk kelas dengan jumlah < target, generate augmented images secara
          berulang dari gambar yang ada hingga mencapai target count
        - Untuk kelas dengan jumlah >= target, tidak dilakukan augmentasi
        - Augmentasi HANYA pada split train; valid & test tidak disentuh

    Penamaan file augmentasi:
        {nama_file_asli}_aug{index:04d}.jpg
        Contoh: corn_rust_001_aug0023.jpg

    Parameters
    ----------
    split_dir : str
        Path direktori hasil split (berisi subdirektori 'train/').
    target_per_class : int
        Target jumlah gambar per kelas setelah augmentasi (default: 2000).
    seed : int
        Random seed untuk reprodusibilitas.

    Returns
    -------
    dict
        Ringkasan per kelas: jumlah gambar asli, augmentasi, dan total.
    """
    random.seed(seed)
    np.random.seed(seed)

    train_dir = Path(split_dir) / "train"

    if not train_dir.exists():
        raise FileNotFoundError(
            f"Direktori train tidak ditemukan: {train_dir}\n"
            f"Pastikan split_dataset() sudah dijalankan terlebih dahulu."
        )

    class_dirs = sorted([d for d in train_dir.iterdir() if d.is_dir()])

    if not class_dirs:
        raise ValueError(f"Tidak ada subdirektori kelas di: {train_dir}")

    # ── Hitung jumlah gambar per kelas ────────────────────────────────────────
    class_files  = {}
    class_counts = {}

    for class_dir in class_dirs:
        # Gunakan helper get_image_files untuk mencakup semua ekstensi
        files = get_image_files(class_dir)
        class_files[class_dir.name]  = files
        class_counts[class_dir.name] = len(files)

    print(f"\n{'='*60}")
    print(f"  AUGMENTASI DATA TRAIN (FIXED TARGET PER KELAS)")
    print(f"{'='*60}")
    print(f"  Target per kelas : {target_per_class} gambar")
    print(f"  Random seed      : {seed}")
    print(f"{'='*60}\n")

    summary = {}

    for class_name, files in class_files.items():
        current_count = class_counts[class_name]
        needed        = target_per_class - current_count

        if needed <= 0:
            print(
                f"  [SKIP] {class_name}\n"
                f"         {current_count} gambar — sudah memenuhi/melampaui target ({target_per_class})\n"
            )
            summary[class_name] = {
                "original":  current_count,
                "augmented": 0,
                "total":     current_count
            }
            continue

        print(
            f"  [AUG]  {class_name}\n"
            f"         {current_count} → {target_per_class} "
            f"(+{needed} gambar akan digenerate)"
        )

        class_dir  = train_dir / class_name
        aug_count  = 0

        # Ulangi daftar file jika jumlah file < needed (cycling)
        file_cycle = (files * (needed // max(len(files), 1) + 2))[:needed]

        for idx, src_file in enumerate(
            tqdm(file_cycle, desc=f"  Augmenting", unit="img")
        ):
            image = cv2.imread(str(src_file))
            if image is None:
                continue

            aug_image    = apply_random_augmentation(image)
            aug_filename = f"{src_file.stem}_aug{idx:04d}.jpg"
            saved = cv2.imwrite(
                str(class_dir / aug_filename),
                aug_image,
                [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY]
            )
            if saved:
                aug_count += 1

        summary[class_name] = {
            "original":  current_count,
            "augmented": aug_count,
            "total":     current_count + aug_count
        }
        print(
            f"         ✅ {aug_count} gambar augmentasi dibuat "
            f"(total: {current_count + aug_count})\n"
        )

    _print_augmentation_summary(summary, target_per_class)
    return summary

In [ ]:
def _print_augmentation_summary(summary: dict, target_count: int) -> None:
    """Mencetak ringkasan hasil augmentasi ke konsol."""
    print(f"\n{'='*68}")
    print(f"  RINGKASAN AUGMENTASI")
    print(f"{'='*68}")
    print(f"  {'Kelas':<44} {'Asli':>6} {'+Aug':>6} {'Total':>6}")
    print(f"  {'-'*44} {'------':>6} {'------':>6} {'------':>6}")

    for class_name, stats in summary.items():
        short = class_name[:42] + ".." if len(class_name) > 42 else class_name
        print(
            f"  {short:<44} "
            f"{stats['original']:>6} "
            f"{stats['augmented']:>6} "
            f"{stats['total']:>6}"
        )

    print(f"  {'-'*44} {'------':>6} {'------':>6} {'------':>6}")
    orig  = sum(s["original"]  for s in summary.values())
    aug   = sum(s["augmented"] for s in summary.values())
    total = sum(s["total"]     for s in summary.values())
    print(f"  {'TOTAL':<44} {orig:>6} {aug:>6} {total:>6}")
    print(f"{'='*68}\n")
    print(f"  ✅ Augmentasi selesai! Target per kelas: {target_count} gambar")
    print()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FUNGSI EXPORT ZIP
# ─────────────────────────────────────────────────────────────────────────────

def export_to_zip(output_dir: str,
                  zip_output_dir: str = ZIP_OUTPUT_DIR,
                  zip_filename: str | None = ZIP_FILENAME) -> str:
    """
    Mengemas seluruh hasil preprocessing ke dalam satu file ZIP.

    File ZIP akan berisi struktur direktori yang sama dengan output_dir:
        preprocessed_dataset_20240602_153000.zip
        └── Corn_(maize)___Common_rust_/
            ├── image1.jpg
            └── ...
        └── Corn_(maize)___healthy/
            └── ...

    Parameters
    ----------
    output_dir : str
        Path direktori hasil preprocessing yang akan dikemas.
    zip_output_dir : str
        Path direktori tempat file ZIP disimpan.
    zip_filename : str or None
        Nama file ZIP (tanpa ekstensi). Jika None, nama dibuat otomatis
        dengan format 'preprocessed_dataset_YYYYMMDD_HHMMSS'.

    Returns
    -------
    str
        Path lengkap file ZIP yang berhasil dibuat.

    Raises
    ------
    FileNotFoundError
        Jika direktori output_dir tidak ditemukan atau kosong.
    """
    output_path = Path(output_dir)

    if not output_path.exists():
        raise FileNotFoundError(
            f"Direktori hasil preprocessing tidak ditemukan: {output_dir}"
        )

    # Kumpulkan semua file gambar dari output_dir (semua ekstensi)
    all_files = []
    for ext in IMAGE_EXTENSIONS:
        all_files.extend(output_path.rglob(f"*{ext}"))
    # Hapus duplikat
    all_files = list({f.resolve(): f for f in all_files}.values())

    if not all_files:
        raise FileNotFoundError(
            f"Tidak ada file gambar di direktori output: {output_dir}"
        )

    # Tentukan nama file ZIP
    if zip_filename is None:
        timestamp    = datetime.now().strftime("%Y%m%d_%H%M%S")
        zip_filename = f"preprocessed_dataset_{timestamp}"

    zip_dir  = Path(zip_output_dir)
    zip_dir.mkdir(parents=True, exist_ok=True)
    zip_path = str(zip_dir / f"{zip_filename}.zip")

    print(f"\n{'='*60}")
    print(f"  MEMBUAT FILE ZIP")
    print(f"{'='*60}")
    print(f"  Source  : {output_dir}")
    print(f"  Output  : {zip_path}")
    print(f"  Total   : {len(all_files)} file gambar")
    print(f"{'='*60}\n")

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for file_path in tqdm(all_files, desc="  Compressing", unit="file"):
            # Path di dalam ZIP relatif terhadap output_dir
            arcname = file_path.relative_to(output_path)
            zf.write(file_path, arcname)

    zip_size_mb = os.path.getsize(zip_path) / (1024 * 1024)

    print(f"\n  ✅ ZIP berhasil dibuat!")
    print(f"  📦 Ukuran file : {zip_size_mb:.1f} MB")
    print(f"  📁 Lokasi      : {zip_path}\n")

    return zip_path

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# EKSEKUSI PIPELINE PREPROCESSING
# ─────────────────────────────────────────────────────────────────────────────

# ── Step 1–4: Pipeline preprocessing (BG Removal → CLAHE → Resize → Norm) ────
prep_summary = preprocess_dataset(
    input_dir  = INPUT_DIR,
    output_dir = OUTPUT_DIR
)

# ── Step 5: Split dataset stratified 80 : 10 : 10 ────────────────────────────
split_summary = split_dataset(
    preprocessed_dir = OUTPUT_DIR,
    split_dir        = SPLIT_OUTPUT_DIR
)

# ── Step 6: Augmentasi data train (target tetap 2000 per kelas) ───────────────
aug_summary = augment_train_data(
    split_dir        = SPLIT_OUTPUT_DIR,
    target_per_class = AUG_TARGET_PER_CLASS
)

# ── Step 7: Buat ZIP dari hasil split + augmentasi ────────────────────────────
if prep_summary["total_failed"] == 0:
    export_to_zip(
        output_dir     = SPLIT_OUTPUT_DIR,
        zip_output_dir = ZIP_OUTPUT_DIR,
        zip_filename   = ZIP_FILENAME
    )
else:
    print(
        f"  ⚠️  ZIP tidak dibuat karena ada {prep_summary['total_failed']} "
        f"gambar yang gagal pada tahap preprocessing.\n"
        f"  Perbaiki error di atas lalu jalankan ulang, atau panggil\n"
        f"  export_to_zip(SPLIT_OUTPUT_DIR) secara manual."
    )